# One building — monitoring with Structured Streaming

Four queries over the doors of one building, in the order the lecture builds
them: look at the stream, give it a schema, filter it, count it, put the count in
a window, and then join it against something that does not move. And then a
fifth, which is the rule of lecture 7 written without a `->`.

**Open both notebooks side by side, with `1_building_simulator.ipynb` on the LEFT
and this one on the RIGHT.** It reads what that one produces, so the two take
turns, and the arrows do not mean anything until the windows are in that order:

* a heading with **❮❮** says *go back to the notebook on the left*;
* a heading with **❯❯**, in that notebook, says *come to this one on the right*.

Every section asks for a prediction first, and the notebook runs **downwards
only**: a reading that has changed is taken again further down, next to the
question that changed it, never by scrolling back up.

---
## How to run the two notebooks

Start `1_building_simulator.ipynb` first. The two notebooks run **interleaved**,
and both of them run **downwards only**: no cell above is ever re-run, because a
reading that changes is taken again further down, where the change is the point.

1. the simulator, from the top down to and including the cell that defines
   `send(...)` — the last one before *1) The morning*;
2. **this notebook**, from the top to the `SELECT` of `Q.9.1`. It prints an empty
   table, which is correct: nothing has been sent;
3. the simulator, *1) The morning, one minute at a time*;
4. **this notebook**, down to and including the two queries of `Q.9.5` and their
   two `SELECT`s. Those are empty too, and that is also correct;
5. the simulator, *2) One floor, four openings*, and then the cell that moves the
   clock on;
6. **this notebook**, the rest of it, straight down: the two `SELECT`s of
   `Q.9.5` again, then section 8;
7. *Clean up* here, which stops the queries, then *Clean up* in the simulator,
   which deletes the topic.

When a table comes back empty where it should not be, the batch has not landed
yet: run **that** cell again, not the ones before it.

## The session, and the Kafka connector

In [1]:
import os, json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, TimestampType)

import glob, pyspark

# the Kafka connector is not part of Spark: it is fetched from Maven, and its
# coordinate has to match *this* Spark and the Scala it was built against.
# Derive both instead of hard-coding them, so the notebook cannot silently
# disagree with the image it is running in.
jars  = glob.glob(pyspark.__path__[0] + '/jars/scala-library-2.13*')
scala = '2.13' if jars else '2.12'
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_%s:%s pyspark-shell'
    % (scala, pyspark.__version__))
print('spark', pyspark.__version__, '/ scala', scala)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("one-building")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark

spark 3.5.0 / scala 2.12


## 1) The stream, before it means anything

`readStream` gives back a DataFrame. Not a different class, not a special
object: **a DataFrame**, with the same API as the ones you have been writing all
year — and that is the whole idea of Structured Streaming.

**Predict:** what are the columns of a DataFrame that has just been read from
Kafka? How many of them carry your data?

In [2]:
servers = "kafka:9092"
topic   = "building-events"

raw_sdf = (spark.readStream
           .format("kafka")
           .option("kafka.bootstrap.servers", servers)
           .option("subscribe", topic)
           .option("startingOffsets", "earliest")
           .load())

raw_sdf.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



Seven columns, and **not one of them is a door**. `key` and `value` are binary,
`topic`, `partition` and `offset` are where the message was, `timestamp` is when
Kafka received it — the *ingestion* time, which is not the event time. Your data
is the bytes inside `value`, and Spark does not yet know what is in there.

So in what sense is this a stream at all? Ask it:

In [3]:
print("raw_sdf.isStreaming ->", raw_sdf.isStreaming)
print("type(raw_sdf)       ->", type(raw_sdf).__name__)

raw_sdf.isStreaming -> True
type(raw_sdf)       -> DataFrame


`isStreaming` is a **property, not a method** — `raw_sdf.isStreaming`, with no
parentheses. Write `raw_sdf.isStreaming()` and Python will tell you that a bool
is not callable, which is a confusing way to find out that you were right about
everything except the brackets.

It is the only thing that distinguishes this object from a DataFrame read out of
a file. Same class, same methods, one flag — and that flag changes what the
operations below are *allowed* to do, which is what the rest of this notebook is
about.

## 2) Giving the bytes a schema

Structured Streaming will not guess. `from_json` needs to be told what the text
inside `value` looks like, and the schema below is the one the simulator
produces.

**Predict:** after `from_json`, is the result still a streaming DataFrame?

In [4]:
schema = StructType([
    StructField("event",     StringType()),
    StructField("unitId",    StringType()),
    StructField("ts",        TimestampType()),
    StructField("car",       StringType()),
    StructField("floor",     IntegerType()),
    StructField("servedDir", StringType()),
])

events_sdf = (raw_sdf
              .select(from_json(col("value").cast("string"), schema).alias("v"))
              .select("v.*"))

events_sdf.printSchema()
print("events_sdf.isStreaming ->", events_sdf.isStreaming)

root
 |-- event: string (nullable = true)
 |-- unitId: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- car: string (nullable = true)
 |-- floor: integer (nullable = true)
 |-- servedDir: string (nullable = true)

events_sdf.isStreaming -> True


Now the columns are the building's, `ts` is a real timestamp, and the flag is
still `True`: **streaming is a property of the lineage, not of one DataFrame.**
Every DataFrame derived from a streaming source is itself streaming, all the way
down to the sink.

## 3) `Q.9.1` — selection and projection

The simplest thing a query can do: keep some rows, keep some columns. Nothing
here is specific to streams, and that is the point worth making out loud.

**Predict:** how many of the 37 events are `DoorOpened`?

In [5]:
doors_sdf = (events_sdf
             .where(col("event") == "DoorOpened")
             .select("ts", "unitId", "car", "floor", "servedDir"))

q91 = (doors_sdf
       .writeStream.format("memory").outputMode("append")
       .queryName("q91").start())

A streaming query does not return rows, it *starts*. The rows go to the sink,
and here the sink is a table in memory that you can then query with ordinary
SQL. Give the simulator a few seconds and run the next cell — more than once.

In [6]:
spark.sql("SELECT * FROM q91 ORDER BY ts, unitId").show(10, False)
print("rows so far:", spark.table("q91").count())

+---+------+---+-----+---------+
|ts |unitId|car|floor|servedDir|
+---+------+---+-----+---------+
+---+------+---+-----+---------+

rows so far: 0


An empty table is a question, not a failure: either the simulator has not sent
that minute yet, or the batch has not landed. A query that has actually failed
raises, and `q91.exception()` says so.

---
# ❮❮ now the notebook on the left

In `1_building_simulator.ipynb`: run *1) The morning, one minute at a time*. It
sends five minutes, with a pause between them, and takes about ten seconds.

Then come back here, to the cell below this one.

## 4) `Q.9.2` — counting, with no window at all

An aggregation over the whole stream, from the beginning of time to now.

**Predict:** three units, 37 events. What does the count of `EU-003` do over the
five minutes — and what does `outputMode` have to be for this to work?

In [7]:
q92 = (doors_sdf
       .groupBy("unitId").count()
       .writeStream.format("memory").outputMode("complete")
       .queryName("q92").start())

In [11]:
spark.sql("SELECT * FROM q92 ORDER BY unitId").show(truncate=False)

+------+-----+
|unitId|count|
+------+-----+
|EU-001|15   |
|EU-002|10   |
|EU-003|12   |
+------+-----+



`complete` is not a stylistic choice. An unbounded aggregation has no point at
which a group is finished, so there is no row Spark could *append*: every new
event may change any group. The only honest thing the sink can hold is the whole
answer, rewritten each time — which also tells you what this query would cost on
a stream that never ends, and is the reason the next one puts a window under it.

Run the cell above again after the simulator has sent another minute. The
numbers move. Nothing was appended; the table was replaced.

## 5) `Q.9.3` — the same count, per minute of event time

**Predict:** you wrote down five numbers in the simulator — 3, 6, 15, 9, 4. Which
of them should come back, and keyed by what?

In [12]:
q93 = (doors_sdf
       .withWatermark("ts", "2 minutes")
       .groupBy(window(col("ts"), "1 minute").alias("w"), col("unitId"))
       .count()
       .writeStream.format("memory").outputMode("update")
       .queryName("q93").start())

In [15]:
spark.sql("""SELECT w.start AS minute, unitId, count
             FROM q93 ORDER BY minute, unitId""").show(20, False)

+-------------------+------+-----+
|minute             |unitId|count|
+-------------------+------+-----+
|2026-10-14 08:00:00|EU-001|2    |
|2026-10-14 08:00:00|EU-002|1    |
|2026-10-14 08:01:00|EU-001|3    |
|2026-10-14 08:01:00|EU-002|2    |
|2026-10-14 08:01:00|EU-003|1    |
|2026-10-14 08:02:00|EU-001|5    |
|2026-10-14 08:02:00|EU-002|4    |
|2026-10-14 08:02:00|EU-003|6    |
|2026-10-14 08:03:00|EU-001|3    |
|2026-10-14 08:03:00|EU-002|2    |
|2026-10-14 08:03:00|EU-003|4    |
|2026-10-14 08:04:00|EU-001|2    |
|2026-10-14 08:04:00|EU-002|1    |
|2026-10-14 08:04:00|EU-003|1    |
+-------------------+------+-----+



Two things to notice, and the second one is a question, not a fact.

**The window is on `ts`, not on the clock.** These are minutes of the building's
morning, decided by the timestamp inside the message. Send the whole trace in
four seconds and the answer is the same — which is what makes a replay of
yesterday's data give yesterday's answer.

**Count the rows: one per minute and unit, and not one more.** In `update` mode a
group is written out again every time its value changes, so a window *can* appear
several times with a growing count — and here it never does. Work out why before
reading on.

It is the sending pattern. Each minute left the simulator as one burst and was
read as one micro-batch, so each window was computed once, from all of its
events, and reported once. A window would be written twice if its events were
split across two batches, or if one of them arrived after the window had already
been reported — which is what *late* means, and it is the subject of the next
lecture's module rather than of this trace.

That is also why the view below exists. On a run where windows do repeat, summing
this table directly counts the intermediate answers as if they were events:

In [16]:
spark.sql("""CREATE OR REPLACE TEMP VIEW q93_final AS
             SELECT w, unitId, MAX(count) AS count
             FROM q93 GROUP BY w, unitId""")

spark.sql("""SELECT w.start AS minute, SUM(count) AS doors
             FROM q93_final GROUP BY w.start ORDER BY minute""").show(10, False)

+-------------------+-----+
|minute             |doors|
+-------------------+-----+
|2026-10-14 08:00:00|3    |
|2026-10-14 08:01:00|6    |
|2026-10-14 08:02:00|15   |
|2026-10-14 08:03:00|9    |
|2026-10-14 08:04:00|4    |
+-------------------+-----+



Those are your five numbers. If they are not, the simulator has not finished
sending — and summing `q93` instead of `q93_final` is the one mistake that
quietly counts the intermediate updates as if they were events.

The `withWatermark` above is how long Spark agrees to wait for a late event
before it may forget a window. Two minutes here, and nothing in this trace is
late. What happens when something *is* late has a module of its own,
`sss_late-arrivals`, and a slide in this lecture.

## 6) `Q.9.4` — joining the stream against something that does not move

The building has two banks: `EU-001` and `EU-002` serve the lower floors,
`EU-003` the upper ones. That table is three rows and it never changes, so it is
not a stream.

**Predict:** doors per bank per minute. The low bank has two units and the high
bank one — is the low bank simply busier, in every minute?

In [17]:
units = spark.createDataFrame(
    [{"unitId": "EU-001", "bank": "low"},
     {"unitId": "EU-002", "bank": "low"},
     {"unitId": "EU-003", "bank": "high"}])

units.show(truncate=False)
print("units.isStreaming ->", units.isStreaming)

+----+------+
|bank|unitId|
+----+------+
|low |EU-001|
|low |EU-002|
|high|EU-003|
+----+------+

units.isStreaming -> False


In [18]:
q94 = (doors_sdf.join(units, on="unitId")
       .withWatermark("ts", "2 minutes")
       .groupBy(window(col("ts"), "1 minute").alias("w"), col("bank"))
       .count()
       .writeStream.format("memory").outputMode("update")
       .queryName("q94").start())

In [20]:
spark.sql("""CREATE OR REPLACE TEMP VIEW q94_final AS
             SELECT w, bank, MAX(count) AS count
             FROM q94 GROUP BY w, bank""")

spark.sql("""SELECT w.start AS minute, bank, count
             FROM q94_final ORDER BY minute, bank""").show(20, False)

+-------------------+----+-----+
|minute             |bank|count|
+-------------------+----+-----+
|2026-10-14 08:00:00|low |3    |
|2026-10-14 08:01:00|high|1    |
|2026-10-14 08:01:00|low |5    |
|2026-10-14 08:02:00|high|6    |
|2026-10-14 08:02:00|low |9    |
|2026-10-14 08:03:00|high|4    |
|2026-10-14 08:03:00|low |5    |
|2026-10-14 08:04:00|high|1    |
|2026-10-14 08:04:00|low |3    |
+-------------------+----+-----+



One side streams and the other does not, and the join is written exactly as you
would write it over two tables. What makes it work is that the static side can be
read again for every batch: Spark re-reads `units` each time, which is also why
changing it mid-query is a bad idea rather than a feature.

Read the answer, not the totals. The low bank leads in the first minutes and the
high bank overtakes it per unit in minute 2, because the people who came through
the lobby have arrived upstairs. One building has a rush hour, and it moves.

## 7) `Q.9.5` — a stream joined to itself

Lecture 7 wrote this rule in EPL, in five lines:

```
@name('bunching')
select a.car as firstCar, b.car as secondCar, a.floor as floor
from pattern [
  every a=DoorOpened
    -> b=DoorOpened(floor=a.floor, servedDir=a.servedDir, car != a.car)
       where timer:within(20 sec)
];
```

*Two cars opening at the same floor, in the same direction, within twenty
seconds* — two cars serving one call, so the next call waits.

Structured Streaming has no `->`. What it has is a stream that can be joined to
itself, and the rest of this section is about everything you have to say out loud
to get the same answer — and about the one place where you cannot.

Start both queries below, then go to the simulator and run
*2) One floor, four openings*.

### first, the rule written as it reads

Same unit, same floor, same direction, different cars, twenty seconds apart.
Nothing else. **Predict the number of rows before you run it.**

In [21]:
from pyspark.sql.functions import expr

a = doors_sdf.withWatermark("ts", "30 seconds").alias("a")
b = doors_sdf.withWatermark("ts", "30 seconds").alias("b")

q95_naive = (a.join(b, expr("""
            a.unitId     = b.unitId
        AND a.floor      = b.floor
        AND a.servedDir  = b.servedDir
        AND a.car       <> b.car
        AND b.ts BETWEEN a.ts - interval 20 seconds
                     AND a.ts + interval 20 seconds"""))
             .select(col("a.car").alias("firstCar"), col("b.car").alias("secondCar"),
                     col("a.ts").alias("firstTs"),   col("b.ts").alias("secondTs"))
             .writeStream.format("memory").outputMode("append")
             .queryName("q95_naive").start())

### then the same rule, with the one word EPL did not need

In [22]:
q95 = (a.join(b, expr("""
            a.unitId     = b.unitId
        AND a.floor      = b.floor
        AND a.servedDir  = b.servedDir
        AND a.car       <> b.car
        AND b.ts >  a.ts
        AND b.ts <= a.ts + interval 20 seconds"""))
       .select(col("a.car").alias("firstCar"), col("b.car").alias("secondCar"),
               col("a.ts").alias("firstTs"),   col("b.ts").alias("secondTs"))
       .writeStream.format("memory").outputMode("append")
       .queryName("q95").start())

Run the two cells below. **Both tables will be empty**, and that is the first
thing to see: nothing has been sent yet — and even once it is, a joined row in
`append` mode is withheld until the watermark says that no partner can still
arrive.

In EPL, `timer:within(20 sec)` was the business rule *and* the bound on state.
Here they are two decisions, and this is the one that governs *when you are told*.

In [25]:
spark.sql("SELECT * FROM q95_naive ORDER BY firstTs, secondTs").show(20, False)
print("q95_naive:", spark.table("q95_naive").count(), "rows")

+--------+---------+-------+--------+
|firstCar|secondCar|firstTs|secondTs|
+--------+---------+-------+--------+
+--------+---------+-------+--------+

q95_naive: 0 rows


In [26]:
spark.sql("SELECT * FROM q95 ORDER BY firstTs, secondTs").show(20, False)
print("q95:", spark.table("q95").count(), "rows")

+--------+---------+-------+--------+
|firstCar|secondCar|firstTs|secondTs|
+--------+---------+-------+--------+
+--------+---------+-------+--------+

q95: 0 rows


---
# ❮❮ now the notebook on the left

In `1_building_simulator.ipynb`: run section *2) One floor, four openings* —
`send(bunching())` — and then the cell below it, *and now move the clock on*.

Then come back here, to the two cells below this one. **Not** the two above: a
reading that changes is taken again lower down.

In [27]:
spark.sql("SELECT * FROM q95_naive ORDER BY firstTs, secondTs").show(20, False)
print("q95_naive:", spark.table("q95_naive").count(), "rows")

+--------+---------+-------------------+-------------------+
|firstCar|secondCar|firstTs            |secondTs           |
+--------+---------+-------------------+-------------------+
|A       |B        |2026-10-14 08:07:00|2026-10-14 08:07:08|
|A       |B        |2026-10-14 08:07:00|2026-10-14 08:07:18|
|B       |A        |2026-10-14 08:07:08|2026-10-14 08:07:00|
|B       |A        |2026-10-14 08:07:08|2026-10-14 08:07:14|
|A       |B        |2026-10-14 08:07:14|2026-10-14 08:07:08|
|A       |B        |2026-10-14 08:07:14|2026-10-14 08:07:18|
|B       |A        |2026-10-14 08:07:18|2026-10-14 08:07:00|
|B       |A        |2026-10-14 08:07:18|2026-10-14 08:07:14|
+--------+---------+-------------------+-------------------+

q95_naive: 8 rows


In [28]:
spark.sql("SELECT * FROM q95 ORDER BY firstTs, secondTs").show(20, False)
print("q95:", spark.table("q95").count(), "rows")

+--------+---------+-------------------+-------------------+
|firstCar|secondCar|firstTs            |secondTs           |
+--------+---------+-------------------+-------------------+
|A       |B        |2026-10-14 08:07:00|2026-10-14 08:07:08|
|A       |B        |2026-10-14 08:07:00|2026-10-14 08:07:18|
|B       |A        |2026-10-14 08:07:08|2026-10-14 08:07:14|
|A       |B        |2026-10-14 08:07:14|2026-10-14 08:07:18|
+--------+---------+-------------------+-------------------+

q95: 4 rows


### Read the three numbers, not the tables

**Eight, four, and three.**

**A join is symmetric and a followed-by is not.** The first query reports every
pair twice, once in each direction — eight rows for four openings — because
nothing in it says which of the two came first. `b.ts > a.ts` is not a detail of
implementation: it is the arrow of `->`, written by hand. With it, four rows.

**And then the number that is not a matter of syntax.** Lecture 7's pattern, on
these same four openings, produces **three** rows and not four. The pair the join
finds and the pattern does not is `A 08:07:00 -> B 08:07:18`: the *first* opening
against the *last*, eighteen seconds apart.

In EPL, `every a -> b` gives each `a` an instance that ends on the **first** `b`
that qualifies. The instance opened at `08:07:00` closed on `08:07:08` and
stopped existing, so it was not there any more when `08:07:18` arrived. A join
has no instances and no notion of first: it reports **every** pair that satisfies
the predicate.

Four openings look like two bunchings and are not — they are closer together than
the rule's own window. On a busy floor that gap grows with the square of the
openings rather than with their number. *Followed-by takes the first follower; a
join takes all of them.* It is the one difference here that is not about
verbosity, and it is the reason the two languages are not interchangeable.

## 8) Nothing above has finished

Scroll up for a moment, but do not run anything: `Q.9.1` printed an empty table
and `Q.9.2` counted 15, 10 and 12. Those numbers were true when you read them and
they are not true now — six more events have arrived since, four at floor 3 and
two at floor 0 two minutes later.

**Predict** before running the four cells below: how many rows does `Q.9.1` hold
now, and what are the three counts of `Q.9.2`?

In [29]:
print("Q.9.1 rows now:", spark.table("q91").count())
spark.sql("SELECT * FROM q91 ORDER BY ts DESC").show(6, False)

Q.9.1 rows now: 43
+-------------------+------+---+-----+---------+
|ts                 |unitId|car|floor|servedDir|
+-------------------+------+---+-----+---------+
|2026-10-14 08:09:30|EU-001|B  |0    |DOWN     |
|2026-10-14 08:09:00|EU-001|A  |0    |UP       |
|2026-10-14 08:07:18|EU-003|B  |3    |UP       |
|2026-10-14 08:07:14|EU-003|A  |3    |UP       |
|2026-10-14 08:07:08|EU-003|B  |3    |UP       |
|2026-10-14 08:07:00|EU-003|A  |3    |UP       |
+-------------------+------+---+-----+---------+
only showing top 6 rows



In [30]:
spark.sql("SELECT * FROM q92 ORDER BY unitId").show(truncate=False)

+------+-----+
|unitId|count|
+------+-----+
|EU-001|17   |
|EU-002|10   |
|EU-003|16   |
+------+-----+



In [31]:
spark.sql("""CREATE OR REPLACE TEMP VIEW q93_final AS
             SELECT w, unitId, MAX(count) AS count
             FROM q93 GROUP BY w, unitId""")

spark.sql("""SELECT w.start AS minute, SUM(count) AS doors
             FROM q93_final GROUP BY w.start ORDER BY minute""").show(10, False)

+-------------------+-----+
|minute             |doors|
+-------------------+-----+
|2026-10-14 08:00:00|3    |
|2026-10-14 08:01:00|6    |
|2026-10-14 08:02:00|15   |
|2026-10-14 08:03:00|9    |
|2026-10-14 08:04:00|4    |
|2026-10-14 08:07:00|4    |
|2026-10-14 08:09:00|2    |
+-------------------+-----+



In [32]:
spark.sql("""CREATE OR REPLACE TEMP VIEW q94_final AS
             SELECT w, bank, MAX(count) AS count
             FROM q94 GROUP BY w, bank""")

spark.sql("""SELECT w.start AS minute, bank, count
             FROM q94_final ORDER BY minute, bank""").show(20, False)

+-------------------+----+-----+
|minute             |bank|count|
+-------------------+----+-----+
|2026-10-14 08:00:00|low |3    |
|2026-10-14 08:01:00|high|1    |
|2026-10-14 08:01:00|low |5    |
|2026-10-14 08:02:00|high|6    |
|2026-10-14 08:02:00|low |9    |
|2026-10-14 08:03:00|high|4    |
|2026-10-14 08:03:00|low |5    |
|2026-10-14 08:04:00|high|1    |
|2026-10-14 08:04:00|low |3    |
|2026-10-14 08:07:00|high|4    |
|2026-10-14 08:09:00|low |2    |
+-------------------+----+-----+



Two minutes appeared that were not there before — a 7 and a 9 — and no cell was
edited to make that happen. **A streaming query does not finish.** What you read
is where it had got to, and the only reason the numbers above the fold are
different is that you read them earlier.

It is also the reason this notebook never asks you to go back up and run a cell
again. A result that changes under you cannot be kept in one place: it has to be
read again, further down, next to the question that made it change.

## What is deliberately not here

**Late arrivals and watermarks, properly.** They appear twice above — as the
bound that lets a window be forgotten, and as the condition that lets a joined
row be final — and both times as a parameter, not as a subject.
`sss_late-arrivals` is the module for that, with a simulator built to make events
late on purpose.

**The fleet.** One building is three units and two banks. The next lecture asks
what changes when it is 150,000 units in three regions, and the answer is not
"the same code, bigger" in every respect. Finding out which parts scale and which
do not is that lecture.

## Clean up

This stops the six queries. Run it **before** the simulator deletes the topic.

In [33]:
for q in (q91, q92, q93, q94, q95_naive, q95):
    q.stop()
print("stopped:", [q.name for q in spark.streams.active] or "none left")

stopped: none left


---
# ❮❮ now the notebook on the left, one last time

In `1_building_simulator.ipynb`: run its *Clean up*, which deletes the topic.
Then save both notebooks and bring the stack down with `docker-compose down`.